# Autoencoder on Galaxy10

We're replaying a classic experiment from **Hinton & Salakhutdinov, *Science* 2006** — ["Reducing the Dimensionality of Data with Neural Networks"](https://www.science.org/doi/10.1126/science.1127647). They trained a deep MLP autoencoder on MNIST and showed that the learned low-dimensional codes cleanly separated the digit classes, even though the model never saw labels during training. It was a pivotal demonstration that deep networks learn *structured representations*, not just input-output mappings.

We're going to replay that experiment with an astronomical twist. Instead of MNIST digits, we'll use **Galaxy10 SDSS**, a dataset of ~22k galaxy thumbnails labeled with 10 morphological classes (smooth, spiral, edge-on, merging, etc.). Same recipe — MLP encoder + bottleneck + MLP decoder, trained end-to-end with MSE — same question: **do morphologically similar galaxies end up near each other in the latent space?**

Concretely we'll learn:
- an **encoder** $f: \mathbb{R}^{69 \times 69 \times 3} \to \mathbb{R}^{16}$ that compresses each image to a 16-dimensional code,
- a **decoder** $g: \mathbb{R}^{16} \to \mathbb{R}^{69 \times 69 \times 3}$ that reconstructs from the code.

The interesting part isn't pixel-perfect reconstruction (we'd need a much bigger model and a different loss for that). It's the *organization* of the 16-d latent space — visualized via PCA at the end of the notebook.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

## 1. The Galaxy10 dataset

Galaxy10 SDSS is a curated set of ~22k galaxy thumbnails with morphology labels (smooth, spiral, edge-on, merging, etc.). Class counts are heavily imbalanced — the rarest class has only 17 examples. We use 90% for training and hold out 10% for validation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

from galaxy_dataset import load_galaxy10, CLASS_NAMES

data = load_galaxy10(device=device)
print(f"train: {data.train_images.shape[0]:>6,} images "
      f"(shape per image: {tuple(data.train_images.shape[1:])})")
print(f"val:   {data.val_images.shape[0]:>6,} images")

In [ ]:
# One example from each class, for a feel of what the model has to represent.
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for cls in range(len(CLASS_NAMES)):
    matches = (data.train_labels == cls).nonzero(as_tuple=True)[0]
    if matches.numel() == 0:
        axes.flat[cls].axis("off")
        continue
    img = (data.train_images[matches[0]].cpu().numpy() + 1) / 2   # [-1,1] -> [0,1] for display
    ax = axes.flat[cls]
    ax.imshow(img)
    ax.set_title(f"{cls}: {CLASS_NAMES[cls]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. The autoencoder

A pair of MLPs:

- **Encoder**: flatten the image to 14,283 features, pass through Linear+ReLU blocks (widths `(512, 128)`), then a final linear projection to the 16-d latent vector.
- **Decoder**: mirror — linear up from 16 to 128 to 512 to 14,283, followed by `tanh` (matching the target range $[-1, 1]$) and reshape back to $(69, 69, 3)$.

No latent regularization, no skip connections — just a plain bottleneck autoencoder.

In [ ]:
from autoencoder import AutoencoderConfig, build_autoencoder

config = AutoencoderConfig(latent_dim=16, hidden_widths=(512, 128))
encoder, decoder = build_autoencoder(config)
encoder.to(device); decoder.to(device)

n_enc = sum(p.numel() for p in encoder.parameters())
n_dec = sum(p.numel() for p in decoder.parameters())
print(f"encoder: {n_enc:>10,} params")
print(f"decoder: {n_dec:>10,} params")
print(f"total:   {n_enc + n_dec:>10,} params")

## 3. Training

Random shuffled mini-batches per epoch, MSE between input and reconstruction, Adam. Validation MSE is computed over the full held-out set after each epoch.

In [ ]:
# --- Knobs ---
batch_size = 256
n_epochs   = 30
lr         = 1e-3
# ---

optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=lr)
criterion = nn.MSELoss()

n_train = data.train_images.shape[0]
train_losses, val_losses = [], []

pbar = tqdm(range(n_epochs), desc="Training")
for epoch in pbar:
    perm = torch.randperm(n_train, device=device)
    epoch_loss, n_batches = 0.0, 0
    for start in range(0, n_train, batch_size):
        idx = perm[start:start + batch_size]
        batch = data.train_images[idx]
        recon = decoder(encoder(batch))
        loss = criterion(recon, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    train_losses.append(epoch_loss / n_batches)

    with torch.no_grad():
        val_loss = criterion(decoder(encoder(data.val_images)), data.val_images).item()
    val_losses.append(val_loss)

    pbar.set_postfix(train=f"{train_losses[-1]:.4f}", val=f"{val_loss:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, "-o", label="train", markersize=4)
ax.plot(val_losses,   "-o", label="val",   markersize=4)
ax.set_xlabel("epoch"); ax.set_ylabel("MSE loss")
ax.set_yscale("log")
ax.set_title(f"Autoencoder training — latent_dim={config.latent_dim}")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Reconstruction quality

Top row: original validation images. Bottom row: reconstructions through the 16-d bottleneck. They won't be pixel-perfect — that's expected at a compression ratio of ~14283 / 16 ≈ 900× — but the morphology should be recognizable.

In [ ]:
n_show = 8
with torch.no_grad():
    sample = data.val_images[:n_show]
    recon  = decoder(encoder(sample))
orig_np  = ((sample.cpu().numpy() + 1) / 2).clip(0, 1)
recon_np = ((recon.cpu().numpy()  + 1) / 2).clip(0, 1)

fig, axes = plt.subplots(2, n_show, figsize=(2 * n_show, 4.5))
for i in range(n_show):
    axes[0, i].imshow(orig_np[i]);  axes[0, i].axis("off")
    axes[1, i].imshow(recon_np[i]); axes[1, i].axis("off")
fig.text(0.01, 0.75, "original",       rotation=90, va="center", fontsize=11)
fig.text(0.01, 0.27, "reconstruction", rotation=90, va="center", fontsize=11)
plt.tight_layout(rect=(0.03, 0, 1, 1))
plt.show()

## 5. The latent space

This is the payoff cell — the Hinton & Salakhutdinov moment, restaged on galaxies. Encode every training image to a 16-d code and project to 2D two ways:

- **PCA** (linear): the two directions of maximum variance in latent space. Honest about distances, but if class structure isn't aligned with the top variance directions it'll wash out.
- **t-SNE** (nonlinear): preserves *local neighborhoods* — points that are nearby in 16-d stay nearby in 2-d. Distances and cluster *sizes* aren't meaningful, but cluster *membership* usually is. Slower (~30 s on ~20k points), and the result is stochastic.

Color each point by its true class label — the autoencoder never saw labels during training, so any clustering by morphology is **emergent structure** in the latent space. If morphologically similar galaxies (e.g. all the edge-on classes) land near each other in t-SNE, the autoencoder has learned a representation that aligns with how humans categorize galaxies.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Encode all training images to z-space.
with torch.no_grad():
    z = encoder(data.train_images).cpu().numpy()    # (N_train, 16)
labels_np = data.train_labels.cpu().numpy()

# PCA: linear, exact, fast.
pca = PCA(n_components=2)
z_pca = pca.fit_transform(z)
print(f"PCA explains {pca.explained_variance_ratio_.sum():.1%} of latent variance")

# t-SNE: nonlinear, stochastic, ~30s. init='pca' + learning_rate='auto' are the
# modern defaults that avoid the bad-local-minima failure modes of older t-SNE.
print("Running t-SNE...")
z_tsne = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=0,
).fit_transform(z)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax, embed, name in [(axes[0], z_pca, "PCA"), (axes[1], z_tsne, "t-SNE")]:
    for cls in range(len(CLASS_NAMES)):
        mask = labels_np == cls
        ax.scatter(embed[mask, 0], embed[mask, 1], s=4, alpha=0.5,
                   label=f"{cls}: {CLASS_NAMES[cls]} (n={mask.sum()})")
    ax.set_xlabel(f"{name} 1"); ax.set_ylabel(f"{name} 2")
    ax.set_title(f"Latent space — {name}")

axes[1].legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=9, markerscale=2)
plt.tight_layout()
plt.show()